# Entraînement et évaluation — Analyse de trafic (UA-DETRAC) — Kaggle

**Avant de lancer** (panneau de droite du notebook Kaggle) :
- *Session options → Accelerator* : **GPU T4 x2** (ou P100)
- *Session options → Internet* : **On** (nécessite un compte vérifié par téléphone)
- *Input → Add Input* : chercher **UA-DETRAC** et ajouter une version qui contient les **annotations XML**
  (dossier `DETRAC-Train-Annotations-XML` ou fichiers `MVI_xxxxx.xml`) — elles contiennent les identifiants de suivi.

Pour garder les résultats : *Save Version → Save & Run All (Commit)*. Tout ce qui est écrit dans `/kaggle/working`
est conservé dans l'onglet *Output* de la version.

Étapes : 1. Environnement · 2. Dataset · 3. Fine-tuning · 4. Éval détection · 5. Suivi · 6. Éval suivi (TrackEval)

> **Partie 6 :** chaque erreur rencontrée → copier le message complet dans `docs/errors_log.md` (+ capture).

## 1. Environnement

In [ ]:
!nvidia-smi
!pip -q install ultralytics lap
import ultralytics, torch, sys, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.version.cuda,
      "| ultralytics", ultralytics.__version__)

In [ ]:
# Dépôt GitHub du projet
REPO = 'https://github.com/jozefelmanga/traffic-tracking.git'
!git clone -q $REPO /kaggle/working/traffic-tracking
%cd /kaggle/working/traffic-tracking
!git log -1 --format='commit %h — %s'
WORK = '/kaggle/working'

## 2. Dataset UA-DETRAC

Le dataset ajouté est monté en lecture seule dans `/kaggle/input/`. La cellule suivante retrouve
automatiquement le dossier des XML et celui des images, quelle que soit la version choisie.

In [ ]:
import glob, os
xmls = glob.glob('/kaggle/input/**/MVI_*.xml', recursive=True)
img_dirs = [d for d in glob.glob('/kaggle/input/**/MVI_*', recursive=True)
            if os.path.isdir(d) and glob.glob(d + '/img*.jpg')]
assert xmls, "Aucun MVI_*.xml trouvé : choisissez une version du dataset avec les annotations XML"
assert img_dirs, "Aucun dossier MVI_xxxxx/img*.jpg trouvé"

# On ne garde que les séquences d'entraînement. Attention : trier les chemins ne suffit pas,
# « Test » passe avant « Train » dans l'ordre alphabétique (bug rencontré, cf. docs/errors_log.md).
xml_dirs = sorted({os.path.dirname(p) for p in xmls})
train_dirs = [d for d in xml_dirs if 'train' in os.path.basename(d).lower()]
assert train_dirs, f"Aucun dossier d'annotations « Train » parmi : {xml_dirs}"
XML = train_dirs[0]
TEST_XML = next((d for d in xml_dirs if 'test' in os.path.basename(d).lower()), None)  # pour plus tard
print('Dossiers XML trouvés :', *xml_dirs, sep='\n  ')
xml_seqs = {os.path.basename(p)[:-4] for p in glob.glob(XML + '/MVI_*.xml')}
roots = {}
for d in img_dirs:
    if os.path.basename(d) in xml_seqs:
        roots.setdefault(os.path.dirname(d), 0); roots[os.path.dirname(d)] += 1
IMAGES = max(roots, key=roots.get)
print('XML    :', XML, f'({len(xml_seqs)} séquences)')
print('IMAGES :', IMAGES, f'({roots[IMAGES]} séquences avec images)')

In [ ]:
!python scripts/convert_detrac.py --images "$IMAGES" --xml "$XML" --out data/detrac_yolo \
    --stride 5 --val-ratio 0.15 --mot-gt
!echo "Séquences de validation :" && cat data/detrac_yolo/val_sequences.txt | tr '\n' ' '
!echo && du -sh data/detrac_yolo

In [ ]:
# Contrôle visuel : on redessine quelques labels pour vérifier la conversion
import cv2, random, matplotlib.pyplot as plt
NAMES = ['car', 'bus', 'van', 'others']
imgs = random.sample(glob.glob('data/detrac_yolo/images/train/*.jpg'), 4)
fig, axs = plt.subplots(2, 2, figsize=(16, 9))
for ax, p in zip(axs.ravel(), imgs):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)   # OpenCV lit en BGR
    H, W = im.shape[:2]
    for line in open(p.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split('\n'):
        if not line: continue
        c, cx, cy, w, h = map(float, line.split())
        x1, y1 = int((cx - w/2) * W), int((cy - h/2) * H)
        cv2.rectangle(im, (x1, y1), (int(x1 + w*W), int(y1 + h*H)), (255, 0, 0), 2)
        cv2.putText(im, NAMES[int(c)], (x1, y1 - 3), 0, 0.5, (255, 0, 0), 1)
    ax.imshow(im); ax.set_title(os.path.basename(p)); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{WORK}/check_labels.jpg', dpi=80)

## 3. Fine-tuning de YOLO11

Transfer learning depuis les poids COCO. Perte totale d'Ultralytics :
`L = λ_box · L_CIoU + λ_cls · L_BCE + λ_dfl · L_DFL` (λ par défaut : 7.5 / 0.5 / 1.5).

In [ ]:
# Reprise : si l'Output d'une version précédente est ajouté en Input (Add Input → Notebook Output),
# on récupère les poids et les sorties de suivi déjà calculées au lieu de tout relancer (~1 h gagnée).
import shutil
prev = sorted(glob.glob('/kaggle/input/**/runs/yolo11n_detrac/weights/best.pt', recursive=True))
if prev:
    prev_runs = prev[0].split('/yolo11n_detrac/')[0]
    shutil.copytree(prev_runs, f'{WORK}/runs', dirs_exist_ok=True)
    prev_trk = os.path.join(os.path.dirname(prev_runs), 'trackeval', 'trackers')
    if os.path.isdir(prev_trk):
        shutil.copytree(prev_trk, f'{WORK}/trackeval/trackers', dirs_exist_ok=True)
    print('Reprise depuis', prev_runs, '→ entraînement sauté')
SKIP_TRAIN = bool(prev)

In [ ]:
from ultralytics import YOLO
if not SKIP_TRAIN:
    model = YOLO('yolo11n.pt')
    model.train(
        data='configs/detrac.yaml',
        epochs=30, imgsz=640, batch=32, patience=8, workers=4,
        device=0,                             # T4 x2 : device=[0, 1] (DDP)
        fliplr=0.5, mosaic=1.0, hsv_v=0.4,    # augmentations : retournement, mosaïque, luminosité (jour/nuit)
        project=f'{WORK}/runs', name='yolo11n_detrac', exist_ok=True,
    )

In [ ]:
# Reprise si la session a été coupée (les poids sont dans l'Output de la version précédente) :
# model = YOLO(f'{WORK}/runs/yolo11n_detrac/weights/last.pt'); model.train(resume=True)

## 4. Évaluation détection

In [ ]:
best = f'{WORK}/runs/yolo11n_detrac/weights/best.pt'
metrics = YOLO(best).val(data='configs/detrac.yaml', imgsz=640, split='val',
                         project=f'{WORK}/runs', name='val', exist_ok=True)
print(f"mAP50 = {metrics.box.map50:.3f}   mAP50-95 = {metrics.box.map:.3f}")
for i, n in enumerate(NAMES):
    print(f"  {n:7s} AP50 = {metrics.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    p = f'{WORK}/runs/yolo11n_detrac/{f}'
    if os.path.exists(p): display(Image(p, width=900))

## 5. Suivi sur les séquences de validation

Le mAP de la baseline COCO n'est pas directement comparable (COCO n'a pas la classe « van »).
La comparaison baseline / fine-tuné se fait donc sur le **suivi**, indépendamment de la classe (HOTA, MOTA, IDF1).

In [ ]:
import subprocess, time
seqs = open('data/detrac_yolo/val_sequences.txt').read().split()
configs = {
    'baseline_bytetrack': ('yolo11n.pt', 'bytetrack.yaml'),
    'ft_bytetrack':       (best, 'bytetrack.yaml'),
    'ft_bytetrack_tuned': (best, 'configs/trackers/bytetrack_tuned.yaml'),
    'ft_botsort_tuned':   (best, 'configs/trackers/botsort_tuned.yaml'),
}
for name, (w, trk) in configs.items():
    t0 = time.time()
    for s in seqs:
        out = f'{WORK}/trackeval/trackers/{name}/data/{s}.txt'
        if os.path.exists(out): continue
        subprocess.run(['python', 'scripts/track_video.py', '--video', f'{IMAGES}/{s}/img%05d.jpg',
                        '--weights', w, '--tracker', trk, '--imgsz', '640', '--device', '0',
                        '--out', f'{WORK}/tracking/{name}/{s}', '--mot-out', out], check=True)
    print(f'{name}: {time.time() - t0:.0f} s')

## 6. Évaluation du suivi (TrackEval)

In [ ]:
if not os.path.isdir(f'{WORK}/TrackEval'):
    !git clone -q https://github.com/JonathonLuiten/TrackEval {WORK}/TrackEval
# TrackEval utilise np.float / np.int, supprimés depuis numpy 1.24
!grep -rlE "np\.(float|int|bool)\b" {WORK}/TrackEval/trackeval | xargs -r sed -i -E 's/np\.(float|int|bool)\b/\1/g'
sys.path.insert(0, f'{WORK}/TrackEval')
import trackeval

# Appel par l'API Python plutôt que par run_mot_challenge.py : en ligne de commande,
# --SEQMAP_FILE est transmis comme une liste → TypeError (bug rencontré, cf. docs/errors_log.md).
open(f'{WORK}/trackeval/seqmap.txt', 'w').write('name\n' + '\n'.join(seqs) + '\n')

def evaluate(gt_folder, trackers_folder):
    ev_cfg = trackeval.Evaluator.get_default_eval_config()
    ev_cfg.update(USE_PARALLEL=False, PRINT_CONFIG=False, PLOT_CURVES=False)
    ds_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    ds_cfg.update(GT_FOLDER=os.path.abspath(gt_folder), TRACKERS_FOLDER=trackers_folder,
                  SEQMAP_FILE=f'{WORK}/trackeval/seqmap.txt', GT_LOC_FORMAT='{gt_folder}/{seq}/gt/gt.txt',
                  SKIP_SPLIT_FOL=True, DO_PREPROC=False, PRINT_CONFIG=False)
    res, _ = trackeval.Evaluator(ev_cfg).evaluate(
        [trackeval.datasets.MotChallenge2DBox(ds_cfg)],
        [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()])
    return res['MotChallenge2DBox']

# (a) évaluation brute
res_raw = evaluate('data/detrac_yolo/mot_gt', f'{WORK}/trackeval/trackers')

### Zones ignorées (protocole officiel UA-DETRAC)

UA-DETRAC n'annote pas les véhicules situés dans les *ignored regions* (souvent la route au loin).
Le modèle fine-tuné les détecte quand même → ils sont comptés comme faux positifs et font chuter le MOTA.
Comme le protocole officiel, on retire les boîtes couvertes à plus de 50 % par ces zones, dans la vérité
terrain **et** dans les sorties des trackers, puis on réévalue.

In [ ]:
!python scripts/filter_ignored.py --gt data/detrac_yolo/mot_gt --trackers {WORK}/trackeval/trackers \
    --out-gt data/detrac_yolo/mot_gt_ign --out-trackers {WORK}/trackeval/trackers_ign
# (b) évaluation sans les zones ignorées
res_ign = evaluate('data/detrac_yolo/mot_gt_ign', f'{WORK}/trackeval/trackers_ign')

In [ ]:
# Tableau récapitulatif pour les slides (brut et sans zones ignorées)
import pandas as pd
rows = []
for protocol, res in [('brut', res_raw), ('sans zones ignorées', res_ign)]:
    for name, r in res.items():
        m = r['COMBINED_SEQ']['pedestrian']      # « pedestrian » = classe unique, ici les véhicules
        rows.append({'protocole': protocol, 'config': name,
                     'HOTA': 100 * m['HOTA']['HOTA'].mean(),
                     'MOTA': 100 * m['CLEAR']['MOTA'],
                     'IDF1': 100 * m['Identity']['IDF1'],
                     'IDSW': int(m['CLEAR']['IDSW']),
                     'FP': int(m['CLEAR']['CLR_FP']), 'FN': int(m['CLEAR']['CLR_FN'])})
summary = pd.DataFrame(rows).sort_values(['protocole', 'HOTA'], ascending=[True, False]).round(1)
summary.to_csv(f'{WORK}/tracking_summary.csv', index=False)
summary

In [ ]:
# Vidéo de démonstration sur une séquence de validation (à télécharger depuis l'onglet Output)
s = seqs[0]
!python scripts/track_video.py --video "$IMAGES/{s}/img%05d.jpg" --weights {best} \
    --tracker configs/trackers/bytetrack_tuned.yaml --scene configs/scene_example.json \
    --imgsz 640 --device 0 --out {WORK}/demo_{s}
!cp {best} {WORK}/best.pt && ls -lh {WORK}

In [ ]:
# Vidéos d'entrée pour la démo en direct (H.264, lisibles dans le navigateur) :
# une séquence facile (jour, trafic fluide) et une difficile (la plus mauvaise en HOTA).
os.makedirs(f'{WORK}/demo_inputs', exist_ok=True)
for s in ['MVI_20035', 'MVI_40752']:
    !ffmpeg -y -loglevel error -framerate 25 -i "$IMAGES/{s}/img%05d.jpg" -c:v libx264 -pix_fmt yuv420p -crf 23 {WORK}/demo_inputs/{s}.mp4
!ls -lh {WORK}/demo_inputs